# Import Libraries

In [1]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

# Sample Text Dataset

In [2]:
text = '''
deep learning is a part of artificial intelligence
deep learning uses neural networks
artificial intelligence is transforming the word
machine learning and deep learning are important
deep lstm models improve sequence learning
natural language processing uses deep learning
'''

# Tokenization

In [4]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1
print('Total Words:', total_words)

Total Words: 26


# Create Input Sequences

In [5]:
input_sequences = []
for line in text.split('\n'):
  token_list = tokenizer.texts_to_sequences([line])[0]
  for i in range(1, len(token_list)):
    n_gram_sequence = token_list[:i+1]
    input_sequences.append(n_gram_sequence)

# Padding Sequences

In [6]:
max_seq_len = max([len(seq) for seq in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre'))

# Split input and Output

In [7]:
X = input_sequences[:,:-1]
y = input_sequences[:,-1]

# One-Hot encoding
y = to_categorical(y,num_classes = total_words)

# Build Deep LSTM Model

In [15]:
model = Sequential()
# Embedding Layer
model.add(Embedding(input_dim = total_words, output_dim=64,input_length= max_seq_len))
# LSTM Layer
model.add(LSTM(128, return_sequences=True))
# Second LSTM Layer
model.add(LSTM(128))
# Output Layer
model.add(Dense(total_words, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


# Compile Model

In [10]:
model.compile(
    loss = 'categorical_crossentropy',
    optimizer = 'adam',
    metrics = ['accuracy']
)

# Model Summary

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Train Model

In [16]:
model.compile(
    loss = 'categorical_crossentropy',
    optimizer = 'adam',
    metrics = ['accuracy']
)
model.fit(X,y,epochs=100,verbose=1)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.0312 - loss: 3.2579
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.1875 - loss: 3.2511
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.1875 - loss: 3.2436
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.1875 - loss: 3.2345
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - accuracy: 0.1875 - loss: 3.2229
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.1875 - loss: 3.2078
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.1875 - loss: 3.1879
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.1875 - loss: 3.1617
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.1875 - loss: 3.1281
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.1875 - loss: 3.0874
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.1875 - loss: 3.0461
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.1875 - los

# Predict Next Multiple Words

In [18]:
def predict_next_words(model, tokenizer, text, max_seq_len, n_words):
  for _ in range(n_words):
    # Convert text into token sequence
    token_list = tokenizer.texts_to_sequences([text])[0]
    # Padding
    token_list = pad_sequences([token_list],
                               maxlen = max_seq_len -1,
                               padding = 'pre')
    # Predict next word
    predicted = np.argmax(model.predict(token_list, verbose = 0), axis =-1) [0]
    output_word = ''

    # Convert index to word
    for word, index in tokenizer.word_index.items():
      if index == predicted:
        output_word = word
        break
    # Append predicted word
    text += ' ' + output_word
  return text

# Next Word prediction

In [24]:
seed_text = 'machine learning'
predicted_text = predict_next_words(model, tokenizer, seed_text, max_seq_len, 5)
print(predicted_text)

print('\nInput :', seed_text)
print('Predicted Sentence :', predicted_text)
#

machine learning uses deep learning are important

Input : machine learning
Predicted Sentence : machine learning uses deep learning are important
